In [1]:
!pip install -q datasets torch evaluate tokenizers transformers rouge_score sacrebleu

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 13.8 MB/s eta 0:00:00


# Dataset

In [2]:
from datasets import load_dataset

dataset = load_dataset("knkarthick/dialogsum")
dataset

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/4.65k [00:00<?, ?B/s]

train.csv: reconstructing file:   0%|          |  0.00B / 11.3MB            

train.csv: downloading bytes:           |  0.00B            

validation.csv:   0%|          | 0.00/442k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12460 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

In [3]:
!pwd

/content


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
dataset['train'][0]

{'id': 'train_0',
 'dialogue': "#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?\n#Person2#: I found it would be a good idea to get a check-up.\n#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.\n#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?\n#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.\n#Person2#: Ok.\n#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?\n#Person2#: Yes.\n#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.\n#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.\n#Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.\n#Person2#: Ok, thanks doctor.",
 'summary': "Mr. Smith'

In [6]:
import pandas as pd
df = dataset['train'].to_pandas()

df.head()

,id,dialogue,summary,topic
0,train_0,"#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. ...","Mr. Smith's getting a check-up, and Doctor Haw...",get a check-up
1,train_1,"#Person1#: Hello Mrs. Parker, how have you bee...",Mrs Parker takes Ricky for his vaccines. Dr. P...,vaccines
2,train_2,"#Person1#: Excuse me, did you see a set of key...",#Person1#'s looking for a set of keys and asks...,find keys
3,train_3,#Person1#: Why didn't you tell me you had a gi...,#Person1#'s angry because #Person2# didn't tel...,have a girlfriend
4,train_4,"#Person1#: Watsup, ladies! Y'll looking'fine t...",Malik invites Nikki to dance. Nikki agrees if ...,dance


In [7]:
df.isnull().sum()

,0
id,0
dialogue,0
summary,0
topic,0


In [8]:
df.duplicated().sum()

np.int64(0)

# Tokenizer and preprocessing

In [7]:
dataset = dataset.remove_columns("id")
dataset

DatasetDict({
    train: Dataset({
        features: ['dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

In [8]:
import os
from tokenizers import Tokenizer, pre_tokenizers, trainers, models

VOCAB_SIZE = 20000

tokenizer = Tokenizer(models.WordLevel(unk_token="<unk>"))
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

trainer = trainers.WordLevelTrainer(
    special_tokens=['<pad>', '<unk>', '<bos>', '<eos>'],
    min_frequency=2,
    vocab_size= VOCAB_SIZE
)

In [9]:
def get_data(dataset):
    for row in dataset:
        yield row['dialogue']

        yield row['summary']

        yield row['topic']

tokenizer.train_from_iterator(get_data(dataset['train']), trainer=trainer)
tokenizer.save('tokenizer.json')

In [10]:
from transformers import PreTrainedTokenizerFast

tokenizer = PreTrainedTokenizerFast(
    tokenizer_file='tokenizer.json',
    unk_token='<unk>',
    pad_token='<pad>',
    bos_token='<bos>',
    eos_token='<eos>',
)

In [11]:
len(tokenizer)

18109

In [12]:
tokenizer(["I go home"])


{'input_ids': [[11, 53, 192]], 'attention_mask': [[1, 1, 1]]}

In [13]:
MAX_DIALOGUE_LEN = 256 
MAX_SUMMARY_LEN = 64
MAX_TOPIC_LEN = 25

def preprocess(examples):
    dialogue_text = examples['dialogue']
    summary_text = ['<bos> ' + text + ' <eos>' for text in examples['summary']]
    topic_text = ['<bos> ' + text + ' <eos>' for text in examples['topic']]

    dialogue_encoding = tokenizer(
        dialogue_text,
        padding='max_length',
        truncation=True,
        max_length=MAX_DIALOGUE_LEN
    )
    
    summary_encoding = tokenizer(
        summary_text,
        padding='max_length',
        truncation=True,
        max_length=MAX_SUMMARY_LEN
    )

    topic_encoding = tokenizer(
        topic_text,
        padding='max_length',
        truncation=True,
        max_length=MAX_TOPIC_LEN
    )

    return {
        'input_ids': dialogue_encoding['input_ids'],
        'summary_labels': summary_encoding['input_ids'],
        'topic_labels': topic_encoding['input_ids']
    }

In [14]:
process_dataset = dataset.map(preprocess, batched=True)
process_dataset

Map:   0%|          | 0/12460 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['dialogue', 'summary', 'topic', 'input_ids', 'summary_labels', 'topic_labels'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['dialogue', 'summary', 'topic', 'input_ids', 'summary_labels', 'topic_labels'],
        num_rows: 500
    })
    test: Dataset({
        features: ['dialogue', 'summary', 'topic', 'input_ids', 'summary_labels', 'topic_labels'],
        num_rows: 1500
    })
})

In [15]:
print(process_dataset['train'][0])

{'dialogue': "#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?\n#Person2#: I found it would be a good idea to get a check-up.\n#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.\n#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?\n#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.\n#Person2#: Ok.\n#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?\n#Person2#: Yes.\n#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.\n#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.\n#Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.\n#Person2#: Ok, thanks doctor.", 'summary': "Mr. Smith's getting a check-u

# Model 

In [16]:
from transformers import PreTrainedModel, PretrainedConfig
import torch
import torch.nn as nn

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
device

device(type='cpu')

In [17]:
class Seq2SeqGRUConfig(PretrainedConfig):
    def __init__(self, vocab_size=20000, embedding_dim=128, hidden_size=128, dropout=0.1, loss_alpha=0.5, **kwargs):
        super().__init__(**kwargs) 
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.hidden_size = hidden_size
        self.dropout = dropout
        self.loss_alpha = loss_alpha

In [18]:
class GRU_Encoder(nn.Module):
    def __init__(self, src_vocab_size, embedding_dim, hidden_size, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(src_vocab_size, embedding_dim)
        self.gru = nn.GRU(embedding_dim, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x): # B, L
        x = self.embedding(x) # B, L, E
        x = self.dropout(x)
        output, hidden = self.gru(x)
        
        return output, hidden # B,L,H ; 1,B,H


class GRU_Decoder(nn.Module):
    def __init__(self, tg_vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(tg_vocab_size, embedding_dim)
        self.gru = nn.GRU(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Linear(512, tg_vocab_size)
        )

    def forward(self, x, hidden):
        x = self.embedding(x)
        output, hidden = self.gru(x, hidden)
        output = self.fc(output)
        return output, hidden

In [19]:
class Seq2SeqGRU(PreTrainedModel):
    config_class = Seq2SeqGRUConfig
    def __init__(self, config, tokenizer):
        super().__init__(config)
        self.encoder = GRU_Encoder(
            src_vocab_size=config.vocab_size,
            embedding_dim=config.embedding_dim,
            hidden_size=config.hidden_size,
            dropout=config.dropout
        )

        self.summary_decoder = GRU_Decoder(
            tg_vocab_size= config.vocab_size,
            embedding_dim=config.embedding_dim,
            hidden_size=config.hidden_size
        )

        self.topic_decoder = GRU_Decoder(
            tg_vocab_size= config.vocab_size,
            embedding_dim=config.embedding_dim,
            hidden_size=config.hidden_size
        )
        self.tokenizer = tokenizer
        self.loss_alpha = config.loss_alpha
        self.loss = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
    
    def forward(self, input_ids, summary_labels, topic_labels):
        tg_summary_input = summary_labels[:, :-1] 
        tg_topic_input = topic_labels[:, :-1]
        
        tg_summary_output = summary_labels[:, 1:] 
        tg_topic_output = topic_labels[:, 1:]

        # encoder
        encoder_output, decoder_hidden = self.encoder(input_ids)

        # decoder
        logits_summary, _ = self.summary_decoder(tg_summary_input, decoder_hidden)  # B, L, V
        logits_topic, _ = self.topic_decoder(tg_topic_input, decoder_hidden)        # B, L, V
        
        # loss
        flatten_logits_summary = logits_summary.reshape(-1, logits_summary.size(-1))
        flatten_target_summary = tg_summary_output.reshape(-1)
        
        flatten_logits_topic = logits_topic.reshape(-1, logits_topic.size(-1))
        flatten_target_topic = tg_topic_output.reshape(-1)

        loss_summary = self.loss(flatten_logits_summary, flatten_target_summary)
        loss_topic = self.loss(flatten_logits_topic, flatten_target_topic)
        total_loss =  self.loss_alpha * loss_summary + (1 - self.loss_alpha) * loss_topic

        return {
            "loss": total_loss,
            "loss_summary": loss_summary,
            "loss_topic": loss_topic,
            "logits_summary": logits_summary,
            "logits_topic": logits_topic
        }

In [20]:
config = Seq2SeqGRUConfig(
    vocab_size= len(tokenizer),
    embedding_dim= 128,
    hidden_size= 128,
    dropout=0.1, 
    loss_alpha=0.5
)

model = Seq2SeqGRU(config, tokenizer)
model.to(device)

Seq2SeqGRU(
  (encoder): GRU_Encoder(
    (embedding): Embedding(18109, 128)
    (gru): GRU(128, 128, batch_first=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (summary_decoder): GRU_Decoder(
    (embedding): Embedding(18109, 128)
    (gru): GRU(128, 128, batch_first=True)
    (fc): Sequential(
      (0): Linear(in_features=128, out_features=512, bias=True)
      (1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (2): ReLU()
      (3): Linear(in_features=512, out_features=18109, bias=True)
    )
  )
  (topic_decoder): GRU_Decoder(
    (embedding): Embedding(18109, 128)
    (gru): GRU(128, 128, batch_first=True)
    (fc): Sequential(
      (0): Linear(in_features=128, out_features=512, bias=True)
      (1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (2): ReLU()
      (3): Linear(in_features=512, out_features=18109, bias=True)
    )
  )
  (loss): CrossEntropyLoss()
)

In [21]:
process_dataset

DatasetDict({
    train: Dataset({
        features: ['dialogue', 'summary', 'topic', 'input_ids', 'summary_labels', 'topic_labels'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['dialogue', 'summary', 'topic', 'input_ids', 'summary_labels', 'topic_labels'],
        num_rows: 500
    })
    test: Dataset({
        features: ['dialogue', 'summary', 'topic', 'input_ids', 'summary_labels', 'topic_labels'],
        num_rows: 1500
    })
})

In [22]:
inputs_id = torch.tensor(process_dataset['train'][0]['input_ids']).unsqueeze(0)
summary_label = torch.tensor(process_dataset['train'][0]['summary_labels']).unsqueeze(0)
topic_label =  torch.tensor(process_dataset['train'][0]['topic_labels']).unsqueeze(0)

inputs_id.shape, summary_label.shape, topic_label.shape

(torch.Size([1, 256]), torch.Size([1, 64]), torch.Size([1, 25]))

In [23]:
output = model(inputs_id.to(device), summary_label.to(device), topic_label.to(device))
output, output['logits_summary'].shape, output['logits_topic'].shape

({'loss': tensor(9.7528, grad_fn=<AddBackward0>),
  'loss_summary': tensor(9.8620, grad_fn=<NllLossBackward0>),
  'loss_topic': tensor(9.6435, grad_fn=<NllLossBackward0>),
  'logits_summary': tensor([[[ 0.1882, -0.2038, -0.1114,  ..., -0.3643, -0.3424, -1.0352],
           [-0.3196,  0.1909, -0.2981,  ..., -0.5053, -0.1576, -0.6553],
           [-0.3305,  0.1772,  0.2153,  ..., -0.2890, -0.1116, -0.8773],
           ...,
           [-0.2550, -0.1386, -0.8336,  ...,  0.2871,  0.1172, -0.5774],
           [-0.2553, -0.1390, -0.8343,  ...,  0.2872,  0.1173, -0.5774],
           [-0.2555, -0.1392, -0.8349,  ...,  0.2873,  0.1175, -0.5775]]],
         grad_fn=<ViewBackward0>),
  'logits_topic': tensor([[[-0.7731, -0.1919, -0.3367,  ..., -0.1641,  0.4102, -0.1098],
           [-0.4043, -0.1063,  0.0352,  ...,  0.5397,  0.5606,  0.0473],
           [-0.5285,  0.3229, -0.1723,  ...,  0.2237,  0.5557,  0.2833],
           ...,
           [-0.0250, -0.0605, -0.0366,  ...,  0.2923, -0.5624, -0.26

## Train

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    logging_strategy='epoch',
    eval_strategy='epoch',
    save_strategy='epoch',
    num_train_epochs=50,
    per_device_train_batch_size=256,
    per_device_eval_batch_size=256,
    learning_rate=5e-5,
    save_total_limit=1,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=process_dataset['train'],
    eval_dataset=process_dataset['validation'],
)

In [29]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.942069,4.166524
2,3.927048,4.160438
3,3.913368,4.154477
4,3.900975,4.148069
5,3.887876,4.143418
6,3.875580,4.137931
7,3.864474,4.133332
8,3.852950,4.127788
9,3.842296,4.123183
10,3.832323,4.118818


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2450, training_loss=3.745172972387197, metrics={'train_runtime': 1281.2974, 'train_samples_per_second': 486.226, 'train_steps_per_second': 1.912, 'total_flos': 1.8192343852032e+16, 'train_loss': 3.745172972387197, 'epoch': 50.0})

In [32]:
repo_id = "huynoob1312/dialogsum-multi-task-gru"
# tokenizer.push_to_hub(repo_id)
# model.push_to_hub(repo_id)

# Evaluate

In [24]:
from transformers import AutoTokenizer
import evaluate
from tqdm import tqdm

repo_id = "huynoob1312/dialogsum-multi-task-gru"

In [25]:
tokenizer = AutoTokenizer.from_pretrained(repo_id)

config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/223 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/420k [00:00<?, ?B/s]

In [27]:
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
import torch

config = Seq2SeqGRUConfig.from_pretrained(repo_id)
model = Seq2SeqGRU(config, tokenizer)

weights_path = hf_hub_download(repo_id=repo_id, filename="model.safetensors")
state_dict = load_file(weights_path)
model.load_state_dict(state_dict)
model.to(device)

model.safetensors: reconstructing file:   0%|          |  0.00B /  104MB            

model.safetensors: downloading bytes:           |  0.00B            

Seq2SeqGRU(
  (encoder): GRU_Encoder(
    (embedding): Embedding(18109, 128)
    (gru): GRU(128, 128, batch_first=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (summary_decoder): GRU_Decoder(
    (embedding): Embedding(18109, 128)
    (gru): GRU(128, 128, batch_first=True)
    (fc): Sequential(
      (0): Linear(in_features=128, out_features=512, bias=True)
      (1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (2): ReLU()
      (3): Linear(in_features=512, out_features=18109, bias=True)
    )
  )
  (topic_decoder): GRU_Decoder(
    (embedding): Embedding(18109, 128)
    (gru): GRU(128, 128, batch_first=True)
    (fc): Sequential(
      (0): Linear(in_features=128, out_features=512, bias=True)
      (1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (2): ReLU()
      (3): Linear(in_features=512, out_features=18109, bias=True)
    )
  )
  (loss): CrossEntropyLoss()
)

In [28]:
rouge_metric = evaluate.load('rouge')
bleu_metric = evaluate.load('sacrebleu')

In [29]:
import torch

def generate_text_gru(model, tokenizer, text, max_summary_len=64, max_topic_len=25):
    model.eval()
    device = next(model.parameters()).device
    
    with torch.no_grad():
        input_encoding = tokenizer(
            text, padding='max_length', truncation=True, 
            max_length=256, return_tensors='pt'
        )
        input_ids = input_encoding['input_ids'].to(device)
        
        _, enc_hidden = model.encoder(input_ids)

        summary_ids = [tokenizer.bos_token_id]
        curr_hidden = enc_hidden  
        curr_input = torch.tensor([[tokenizer.bos_token_id]], device=device)
        
        for _ in range(max_summary_len):
            logits, curr_hidden = model.summary_decoder(curr_input, curr_hidden)
            
            next_word = logits[0, -1, :].argmax().item()
            summary_ids.append(next_word)
            
            if next_word == tokenizer.eos_token_id:
                break
                
            curr_input = torch.tensor([[next_word]], device=device)
            
        topic_ids = [tokenizer.bos_token_id]
        curr_hidden = enc_hidden  
        curr_input = torch.tensor([[tokenizer.bos_token_id]], device=device)
        
        for _ in range(max_topic_len):
            logits, curr_hidden = model.topic_decoder(curr_input, curr_hidden)
            
            next_word = logits[0, -1, :].argmax().item()
            topic_ids.append(next_word)
            
            if next_word == tokenizer.eos_token_id:
                break
                
            curr_input = torch.tensor([[next_word]], device=device)
            
    final_summary = tokenizer.decode(summary_ids, skip_special_tokens=True)
    final_topic = tokenizer.decode(topic_ids, skip_special_tokens=True)
    
    return final_summary, final_topic


In [31]:
sample_idx = 10 
test_dialogue = dataset['test'][sample_idx]['dialogue']
real_summary = dataset['test'][sample_idx]['summary']
real_topic = dataset['test'][sample_idx]['topic']

print("========= HỘI THOẠI GỐC =========")
print(test_dialogue)
print("\n")

pred_summary, pred_topic = generate_text_gru(model, tokenizer, test_dialogue)

print("========= KẾT QUẢ CỦA MODEL =========")
print(f"🔹 TOPIC (AI đoán):  {pred_topic}")
print(f"🔹 SUMMARY (AI đoán): {pred_summary}")
print("\n")
print("========= KẾT QUẢ THỰC TẾ =========")
print(f"🔸 TOPIC (Gốc):  {real_topic}")
print(f"🔸 SUMMARY (Gốc): {real_summary}")


========= HỘI THOẠI GỐC =========
#Person1#: Happy Birthday, this is for you, Brian.
#Person2#: I'm so happy you remember, please come in and enjoy the party. Everyone's here, I'm sure you have a good time.
#Person1#: Brian, may I have a pleasure to have a dance with you?
#Person2#: Ok.
#Person1#: This is really wonderful party.
#Person2#: Yes, you are always popular with everyone. and you look very pretty today.
#Person1#: Thanks, that's very kind of you to say. I hope my necklace goes with my dress, and they both make me look good I feel.
#Person2#: You look great, you are absolutely glowing.
#Person1#: Thanks, this is a fine party. We should have a drink together to celebrate your birthday


========= KẾT QUẢ CỦA MODEL =========
🔹 TOPIC (AI đoán):  a new job
🔹 SUMMARY (AI đoán): # Person1 # and # Person2 # are talking about the new job .


========= KẾT QUẢ THỰC TẾ =========
🔸 TOPIC (Gốc):  birthday party
🔸 SUMMARY (Gốc): #Person1# attends Brian's birthday party. Brian thinks #Perso

In [33]:
pred_summaries = []
ref_summaries = []
pred_topics = []
ref_topics = []

test_data = process_dataset['test']
NUM_TEST = len(test_data)

for i in tqdm(range(NUM_TEST)):
    dialogue = test_data[i]['dialogue']
    
    pred_sum, pred_top = generate_text_gru(model, tokenizer, dialogue)
    
    pred_summaries.append(pred_sum)
    pred_topics.append(pred_top)
    
    ref_summaries.append(test_data[i]['summary'])
    ref_topics.append(test_data[i]['topic'])

summary_rouge = rouge_metric.compute(predictions=pred_summaries, references=ref_summaries)
summary_bleu = bleu_metric.compute(predictions=pred_summaries, references=ref_summaries)
topic_bleu = bleu_metric.compute(predictions=pred_topics, references=ref_topics)
topic_rouge = rouge_metric.compute(predictions=pred_topics, references=ref_topics)


print("\n KẾT QUẢ SUMMARY:")
print(f" - ROUGE-1 (Khớp từng từ): {summary_rouge['rouge1'] * 100:.2f}")
print(f" - ROUGE-2 (Khớp cụm 2 từ): {summary_rouge['rouge2'] * 100:.2f}")
print(f" - ROUGE-L (Khớp chuỗi dài nhất): {summary_rouge['rougeL'] * 100:.2f}")
print(f" - Điểm BLEU của Summary: {summary_bleu['score']:.2f} ")

print("\n KẾT QUẢ TOPIC:")
print(f" - Điểm BLEU: {topic_bleu['score']:.2f}")
print(f" - Điểm ROUGE-L của Topic: {topic_rouge['rougeL'] * 100:.2f}")


  0%|          | 2/1500 [00:00<01:43, 14.44it/s]

100%|██████████| 1500/1500 [01:38<00:00, 15.23it/s]



 KẾT QUẢ SUMMARY:
 - ROUGE-1 (Khớp từng từ): 22.36
 - ROUGE-2 (Khớp cụm 2 từ): 3.98
 - ROUGE-L (Khớp chuỗi dài nhất): 18.49
 - Điểm BLEU của Summary: 8.67 

 KẾT QUẢ TOPIC:
 - Điểm BLEU: 0.00
 - Điểm ROUGE-L của Topic: 4.24
